# Synthetic-lethality pilot analysis (Session 26)

Loads `outputs/synthlet/pilot_predictions.csv` produced by `scripts/run_synthlet_pilot.py` and computes:

1. Per-category synthetic-lethality rate (rows where `is_synthetic_lethal == 1`)
2. Top 10 highest-confidence synthetic-lethal predictions across all categories
3. Failure-mode breakdown (which detector signal flagged the joint knockouts)
4. Go / no-go recommendation for the full ~105k-pair screen, decided by the same rules the run script applies inline.

This is **pilot-scale infrastructure validation**, not a finished synthetic-lethality screen. Every flagged pair is a simulator-derived hypothesis that needs wet-lab validation before any biological claim is made.

Decision rule (locked at start of Session 26):

| outcome | action |
|---|---|
| Category B same-pathway rate > 15 % | **HALT** — methodology broken |
| Category C random rate > 10 % | **HALT** — false-positive rate too high |
| Category A paralog rate > 20 % AND categories B + C below their caps | **PROCEED** to full screen |
| All caps respected but Category A < 20 % | **NARROW SCOPE** — full screen but reduced to a hub-genes-vs-all design |

In [ ]:
import pandas as pd
from pathlib import Path

REPO_ROOT = Path('.').resolve()
while not (REPO_ROOT / 'PROJECT_STATUS.md').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
PRED_CSV = REPO_ROOT / 'outputs/synthlet/pilot_predictions.csv'
PAIRS_CSV = REPO_ROOT / 'outputs/synthlet/pilot_pairs.csv'

df = pd.read_csv(PRED_CSV)
print(f'rows: {len(df)}')
print(f'columns: {list(df.columns)}')
df.head()

In [ ]:
# 1. Per-category synthetic lethality rate
g = (
    df.groupby('category')
      .agg(
          n=('is_synthetic_lethal', 'size'),
          n_synthlet=('is_synthetic_lethal', 'sum'),
          n_pair_essential=('pair_essential', 'sum'),
          n_single_a_ess=('single_a_essential', 'sum'),
          n_single_b_ess=('single_b_essential', 'sum'),
      )
)
g['rate_synthlet'] = g['n_synthlet'] / g['n']
g['rate_pair_essential'] = g['n_pair_essential'] / g['n']
print('Per-category breakdown:')
print(g.to_string())

In [ ]:
# 2. Top-10 synthetic-lethal predictions by confidence (cross-category)
synth = df[df['is_synthetic_lethal'] == 1].copy()
synth = synth.sort_values('pair_confidence', ascending=False)
print(f'Total synthetic-lethal calls: {len(synth)}')
if len(synth) > 0:
    cols = [
        'category', 'locus_a', 'locus_b', 'pair_confidence',
        'pair_mode', 'mechanism_summary', 'biological_rationale',
    ]
    print('\nTop 10:')
    for _, r in synth.head(10).iterrows():
        print(f"  {r['category']:24s} {r['locus_a']:18s} x {r['locus_b']:18s} "
              f"conf={r['pair_confidence']:.3f} mode={r['pair_mode']}")
        print(f"      mech: {str(r['mechanism_summary'])[:120]}")
        print(f"      bio:  {str(r['biological_rationale'])[:100]}")

In [ ]:
# 3. Failure-mode breakdown for synthetic-lethal calls
if len(synth) > 0:
    print('synthetic-lethal calls by detector failure mode:')
    print(synth['pair_mode'].value_counts().to_string())
    print('\nby category x mode:')
    pivot = synth.pivot_table(
        index='category', columns='pair_mode',
        values='locus_a', aggfunc='count', fill_value=0,
    )
    print(pivot.to_string())
else:
    print('No synthetic-lethal calls — methodology may be too conservative or simulator lacks signal.')

In [ ]:
# 4. Go/no-go decision
rate_a = float(g.loc['A_paralog', 'rate_synthlet']) if 'A_paralog' in g.index else 0.0
rate_b = float(g.loc['B_same_pathway', 'rate_synthlet']) if 'B_same_pathway' in g.index else 0.0
rate_c = float(g.loc['C_random', 'rate_synthlet']) if 'C_random' in g.index else 0.0
rate_d = float(g.loc['D_transporter_substrate', 'rate_synthlet']) if 'D_transporter_substrate' in g.index else 0.0
rate_e = float(g.loc['E_manual', 'rate_synthlet']) if 'E_manual' in g.index else 0.0

print(f'Category A paralog       : {rate_a:.1%}')
print(f'Category B same-pathway  : {rate_b:.1%}  (must be < 15%)')
print(f'Category C random        : {rate_c:.1%}  (must be < 10%)')
print(f'Category D transporter   : {rate_d:.1%}')
print(f'Category E manual        : {rate_e:.1%}')

if rate_b > 0.15:
    decision = 'HALT'
    rationale = f'Category B (same-pathway) at {rate_b:.1%} exceeds the 15% cap — methodology flags pairs as synthetic-lethal even when knocking out either gene already breaks the pathway. Investigate before scaling.'
elif rate_c > 0.10:
    decision = 'HALT'
    rationale = f'Category C (random) at {rate_c:.1%} exceeds the 10% cap — false-positive rate too high to extrapolate to a 105k-pair screen.'
elif rate_a >= 0.20:
    decision = 'PROCEED'
    rationale = f'Paralog rate {rate_a:.1%} is meaningfully above the random baseline ({rate_c:.1%}) and the same-pathway control ({rate_b:.1%}). Categories separate; full screen is justified.'
elif rate_a > rate_c + 0.05:
    decision = 'NARROW_SCOPE'
    rationale = f'Paralog rate {rate_a:.1%} sits above random ({rate_c:.1%}) but below the 20% threshold; the signal is weaker than hoped. A narrow hub-genes-vs-all screen (~22k pairs) is the proportional next step.'
else:
    decision = 'NARROW_SCOPE'
    rationale = f'Paralog rate {rate_a:.1%} does not meaningfully separate from random ({rate_c:.1%}). The simulator may not carry enough signal to discriminate synthetic-lethal pairs at this dataset size; a narrow screen could still surface high-confidence individual hits.'

print(f'\nDecision: {decision}')
print(f'Rationale: {rationale}')

## Honest framing

- All synthetic-lethal predictions in this notebook are **simulator-derived hypotheses**, not validated findings. None has been tested at the bench in this work.
- The simulator's known limitations (no Layer 5 biomass + division, ~75 % knowledge-based detector weight at v15) carry over to pairwise predictions. A pair flagged synthetic-lethal here may reflect detector noise rather than real compensatory pathway biology.
- Category B (same-pathway sequential pairs) is a deliberate negative control. Its rate must stay low for the methodology to be interpretable.
- If the methodology yields a clean separation, the natural follow-up is a wet-lab double-knockout test in Syn3A on the highest-confidence calls — single-gene knockout protocol is established (Breuer 2019); pairwise via sequential CRISPRi knockdowns is straightforward.